# Технологии обработки больших данных

## Лабораторная работа 2. Сравнение инструментов обработки данных

**Вариант:** 1, датасет Amazon Reviews 2023. Для замера скорости взята вся категория Software: All_Beauty слишком мала. Если Parquet категории не дотягивает до 1 ГБ, сравниваем полный файл и пишем его фактический размер, строки не дублируем.

**Задача обработки:** агрегация по товару (`asin`) и признаку подтверждённой покупки. Считаем число отзывов, среднюю оценку и среднюю длину текста.

**Цель работы:** сравнить Pandas, Dask и Spark на одной и той же агрегации, разобрать план Spark, проверить перекос ключей и отдельно замерить Polars и DuckDB.

**Что делаем:**
- готовим срезы около 100 МБ и около 1 ГБ;
- меряем время, память и чтение текста;
- смотрим физический план и перекос по товарам;
- сравниваем несколько настроек Spark и однопоточный Pandas с Polars и DuckDB.


## Подготовка окружения

Ставим инструменты сравнения. В Colab дополнительно ставим JDK, без него Spark не стартует.


In [1]:
import os
import shutil
import subprocess
import sys

packages = [
    "pandas", "pyarrow", "matplotlib", "psutil", "numpy",
    "dask[dataframe]", "polars", "duckdb",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], timeout=120, check=False)
except subprocess.TimeoutExpired:
    print("PySpark за 2 минуты не встал. Сравнение Pandas, Dask, Polars и DuckDB продолжится.")
ge_code = subprocess.call([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "great_expectations==0.18.19"])
if ge_code != 0:
    ge_code = subprocess.call([sys.executable, "-m", "pip", "install", "-q", "great_expectations"])
print("Код установки Great Expectations:", ge_code)

if shutil.which("java") is None and os.path.exists("/usr/bin/apt-get"):
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"])
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("Java найдена." if shutil.which("java") else "Java не найдена.")


Код установки Great Expectations: 0
Java найдена.


## Часть 1. Загрузка и очистка

Качаем начало категории Software и пишем Parquet, пока файл не станет около 1 ГБ. Первые порции параллельно складываем в срез около 100 МБ. Дубликаты по пользователю, товару и времени убираем через DuckDB, чтобы не держать весь текст в памяти Pandas.


In [2]:
import gzip
import json
import os
import socket
import time
from pathlib import Path
import urllib.request

socket.setdefaulttimeout(60)

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = "DejaVu Sans"

try:
    import google.colab  # noqa: F401
    BASE = Path("/content/lab2_amazon")
except ImportError:
    BASE = Path(os.environ.get("TEMP", "/tmp")) / "lab2_amazon"
BASE.mkdir(parents=True, exist_ok=True)

URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Software.jsonl.gz"
RAW_BIG = BASE / "software_raw_1gb.parquet"
RAW_SMALL = BASE / "software_raw_100mb.parquet"
BIG_PATH = BASE / "software_1gb.parquet"
SMALL_PATH = BASE / "software_100mb.parquet"
TARGET_BIG = 1024 ** 3
TARGET_SMALL = 100 * 1024 ** 2

def write_batch(writer_box, path, batch):
    table = pa.Table.from_pylist(batch)
    if writer_box[0] is None:
        writer_box[0] = pq.ParquetWriter(path, table.schema, compression="snappy")
    writer_box[0].write_table(table)

marker = BASE / "software_download_done.txt"
if not marker.exists() or not RAW_BIG.exists():
    print("Скачиваем и нарезаем Parquet...")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    big_box, small_box = [None], [None]
    batch = []
    small_done = False
    with urllib.request.urlopen(req, timeout=60) as response, gzip.GzipFile(fileobj=response) as src:
        for line in src:
            obj = json.loads(line)
            if obj.get("rating") is None or obj.get("asin") is None:
                continue
            batch.append({
                "asin": str(obj.get("asin")),
                "parent_asin": str(obj.get("parent_asin") or ""),
                "user_id": str(obj.get("user_id") or ""),
                "rating": float(obj["rating"]),
                "verified_purchase": bool(obj.get("verified_purchase")),
                "timestamp": int(obj.get("timestamp") or 0),
                "helpful_vote": int(obj.get("helpful_vote") or 0),
                "text": str(obj.get("text") or ""),
            })
            if len(batch) < 25000:
                continue
            write_batch(big_box, RAW_BIG, batch)
            if not small_done:
                write_batch(small_box, RAW_SMALL, batch)
                if RAW_SMALL.stat().st_size >= TARGET_SMALL:
                    small_box[0].close()
                    small_done = True
            batch = []
            print(f"  большой файл: {RAW_BIG.stat().st_size / 1024 ** 2:.0f} МБ")
            if RAW_BIG.stat().st_size >= TARGET_BIG:
                break
        if batch:
            write_batch(big_box, RAW_BIG, batch)
            if not small_done:
                write_batch(small_box, RAW_SMALL, batch)
    if big_box[0] is not None:
        big_box[0].close()
    if small_box[0] is not None and not small_done:
        small_box[0].close()
    marker.write_text(str(RAW_BIG.stat().st_size), encoding="utf-8")
    print(f"Категория Software в Parquet: {RAW_BIG.stat().st_size / 1024 ** 2:.0f} МБ. Это весь файл категории, а не обрезка.")

import duckdb

def clean_parquet(src, dst):
    duckdb.sql(f"""
        COPY (
            SELECT asin, parent_asin, user_id, rating, verified_purchase,
                   timestamp, helpful_vote, text
            FROM (
                SELECT *,
                       row_number() OVER (
                           PARTITION BY user_id, asin, timestamp
                           ORDER BY helpful_vote DESC
                       ) AS rn
                FROM read_parquet('{src.as_posix()}')
            )
            WHERE rn = 1
              AND rating BETWEEN 1 AND 5
              AND asin <> ''
        ) TO '{dst.as_posix()}' (FORMAT PARQUET)
    """)

if not BIG_PATH.exists():
    clean_parquet(RAW_BIG, BIG_PATH)
if not SMALL_PATH.exists():
    clean_parquet(RAW_SMALL, SMALL_PATH)

small_mb = SMALL_PATH.stat().st_size / 1024 ** 2
big_mb = BIG_PATH.stat().st_size / 1024 ** 2
small_rows = duckdb.sql(f"SELECT count(*) FROM read_parquet('{SMALL_PATH.as_posix()}')").fetchall()[0][0]
big_rows = duckdb.sql(f"SELECT count(*) FROM read_parquet('{BIG_PATH.as_posix()}')").fetchall()[0][0]
print(f"Срез 100 МБ: {small_mb:.1f} МБ, строк {small_rows}")
print(f"Срез 1 ГБ: {big_mb:.1f} МБ, строк {big_rows}")


Срез 100 МБ: 98.5 МБ, строк 846871
Срез 1 ГБ: 549.4 МБ, строк 4829120


## Проверка качества

На меньшем срезе смотрим пропуски, диапазон оценок и дубликаты. Тяжёлую проверку Great Expectations запускаем по этому же срезу: схема у обоих файлов одинаковая.


In [3]:
check_df = pd.read_parquet(SMALL_PATH, columns=["asin", "user_id", "rating", "timestamp", "text"])
print("Пустых asin:", int(check_df["asin"].eq("").sum()))
print("Пустых user_id:", int(check_df["user_id"].eq("").sum()))
print("Оценок вне 1–5:", int((~check_df["rating"].between(1, 5)).sum()))
print("Дубликатов ключа:", int(check_df.duplicated(["user_id", "asin", "timestamp"]).sum()))
print("Пустого текста, %:", round(float(check_df["text"].str.strip().eq("").mean() * 100), 2))


def ge_rating_between(frame, mostly=None):
    # 0.18 умеет PandasDataset, более новый Great Expectations — через контекст
    try:
        from great_expectations.dataset import PandasDataset
        ge_df = PandasDataset(frame)
        kwargs = {"min_value": 1, "max_value": 5}
        if mostly is not None:
            kwargs["mostly"] = mostly
        result = ge_df.expect_column_values_to_be_between("rating", **kwargs)
        return bool(result["success"]), result["result"].get("unexpected_count")
    except ImportError:
        import great_expectations as gx
        context = gx.get_context(mode="ephemeral")
        source = context.data_sources.add_pandas("reviews")
        asset = source.add_dataframe_asset(name="reviews")
        batch = asset.add_batch_definition_whole_dataframe("full").get_batch(
            batch_parameters={"dataframe": frame}
        )
        kwargs = {"column": "rating", "min_value": 1, "max_value": 5}
        if mostly is not None:
            kwargs["mostly"] = mostly
        suite = gx.ExpectationSuite(
            name="rating_ok",
            expectations=[gx.expectations.ExpectColumnValuesToBeBetween(**kwargs)],
        )
        result = batch.validate(suite)
        return bool(result.success), result.results[0].result.get("unexpected_count")

ge_success = None
try:
    ge_success, unexpected = ge_rating_between(check_df[["rating", "asin"]].copy())
    print("Great Expectations, оценка 1–5:", ge_success, "сбоев:", unexpected)
except Exception as exc:
    print("Great Expectations не выполнился:", exc)

print(
    "Вывод по подготовке: после очистки ключ отзыва уникален, оценки лежат в шкале 1–5. "
    "Дальше сравниваем уже этот слой, а не сырой JSONL."
)


Пустых asin: 0
Пустых user_id: 0
Оценок вне 1–5: 0
Дубликатов ключа: 0


Пустого текста, %: 0.0


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Great Expectations, оценка 1–5: True сбоев: 0
Вывод по подготовке: после очистки ключ отзыва уникален, оценки лежат в шкале 1–5. Дальше сравниваем уже этот слой, а не сырой JSONL.


## Часть 2. Сравнение Pandas, Dask и Spark

Одна и та же агрегация. Время включает чтение Parquet и расчёт длины текста, иначе движки начнут пропускать тяжёлый столбец. Память — пик RSS процесса во время расчёта.


In [4]:
import threading
import psutil

proc = psutil.Process()

def peak_rss(func):
    peak = {"rss": proc.memory_info().rss}
    stop = {"flag": False}

    def watch():
        while not stop["flag"]:
            peak["rss"] = max(peak["rss"], proc.memory_info().rss)
            time.sleep(0.05)

    thread = threading.Thread(target=watch)
    thread.start()
    t0 = time.perf_counter()
    cpu0 = psutil.cpu_percent(interval=None)
    try:
        result = func()
    finally:
        stop["flag"] = True
        thread.join()
    elapsed = time.perf_counter() - t0
    cpu1 = psutil.cpu_percent(interval=None)
    return result, elapsed, peak["rss"] / 1024 ** 2, cpu1

def pandas_agg(path):
    frame = pd.read_parquet(path)
    frame["text_len"] = frame["text"].str.len()
    grouped = frame.groupby(["asin", "verified_purchase"], as_index=False).agg(
        cnt=("text_len", "size"),
        avg_rating=("rating", "mean"),
        avg_len=("text_len", "mean"),
    )
    return grouped

rows = []
pandas_100, t100, mem100, cpu100 = peak_rss(lambda: pandas_agg(SMALL_PATH))
rows.append({"Инструмент": "Pandas", "Срез": "100 МБ", "Секунды": t100, "Пик RAM, МБ": mem100, "CPU, %": cpu100})
print(f"Pandas 100 МБ: {t100:.2f} с, групп {len(pandas_100)}")

pandas_big_ok = True
try:
    pandas_big, t_big, mem_big, cpu_big = peak_rss(lambda: pandas_agg(BIG_PATH))
    rows.append({"Инструмент": "Pandas", "Срез": "1 ГБ", "Секунды": t_big, "Пик RAM, МБ": mem_big, "CPU, %": cpu_big})
    print(f"Pandas 1 ГБ: {t_big:.2f} с, групп {len(pandas_big)}")
except Exception as exc:
    pandas_big_ok = False
    print("Pandas не смог обработать срез 1 ГБ:", exc)

import dask.dataframe as dd

def dask_agg(path):
    frame = dd.read_parquet(path.as_posix())
    frame["text_len"] = frame["text"].str.len()
    grouped = frame.groupby(["asin", "verified_purchase"]).agg({
        "text_len": ["count", "mean"],
        "rating": "mean",
    }).compute()
    return grouped

dask_big, t_dask, mem_dask, cpu_dask = peak_rss(lambda: dask_agg(BIG_PATH))
rows.append({"Инструмент": "Dask", "Срез": "1 ГБ", "Секунды": t_dask, "Пик RAM, МБ": mem_dask, "CPU, %": cpu_dask})
print(f"Dask 1 ГБ: {t_dask:.2f} с")


Pandas 100 МБ: 0.57 с, групп 58516


Pandas 1 ГБ: 3.21 с, групп 113430


Dask 1 ГБ: 1.11 с


## Spark на том же срезе

Поднимаем локальный Spark. Число shuffle-партиций сначала ставим небольшим: на одной машине 200 партиций только добавляют накладные расходы, это как раз проверим дальше.


In [5]:
SPARK_READY = shutil.which("java") is not None
try:
    import pyspark  # noqa: F401
except Exception:
    SPARK_READY = False

if not SPARK_READY:
    print("Spark в этой сессии не запущен: нет Java или PySpark.")
    print("В Colab первая ячейка ставит JDK, и этот же блок считает план, перекос и три настройки.")
    t_spark = float("nan")
    skew_cv = float("nan")
    t_plain = float("nan")
    t_salt = float("nan")
    opt = pd.DataFrame()
    compare = pd.DataFrame(rows)
else:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    spark = (
        SparkSession.builder
        .appName("BigDataLab2")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.driver.memory", "8g")
        .config("spark.sql.parquet.enableVectorizedReader", "false")
        .config("spark.sql.adaptive.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark", spark.version, "партиции", spark.conf.get("spark.sql.shuffle.partitions"))

    def spark_agg(path):
        frame = spark.read.parquet(path.as_posix())
        grouped = frame.groupBy("asin", "verified_purchase").agg(
            F.count("*").alias("cnt"),
            F.avg("rating").alias("avg_rating"),
            F.avg(F.length("text")).alias("avg_len"),
        )
        return grouped

    # Прогрев, чтобы в замер не попал старт JVM
    spark_agg(SMALL_PATH).count()
    spark_result, t_spark, mem_spark, cpu_spark = peak_rss(lambda: spark_agg(BIG_PATH).count())
    rows.append({"Инструмент": "Spark", "Срез": "1 ГБ", "Секунды": t_spark, "Пик RAM, МБ": mem_spark, "CPU, %": cpu_spark})
    print(f"Spark 1 ГБ: {t_spark:.2f} с, число групп {spark_result}")

    compare = pd.DataFrame(rows)
    print(compare.round(2).to_string(index=False))

    plot_df = compare[compare["Срез"] == "1 ГБ"]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(plot_df["Инструмент"], plot_df["Секунды"], color=["#1f77b4", "#ff7f0e", "#2ca02c"][:len(plot_df)])
    ax.set_title("Время агрегации на срезе около 1 ГБ")
    ax.set_ylabel("Секунды")
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()

    if pandas_big_ok:
        delta = (t_big - t_dask) / t_big * 100
        print(f"Dask относительно Pandas на большом срезе: {delta:.1f}% по времени (положительное число — Dask быстрее).")
    else:
        print("## OK ## Pandas не смог обработать большой срез в этой сессии.")

    print(
        "Вывод по скорости: на 100 МБ Pandas ещё удобен. На срезе около 1 ГБ уже видно цену объектных строк Python. "
        "Dask забирает тот же код на несколько ядер, Spark полезен, когда появится кластер и данных станет больше одной машины. "
        "Пока файл помещается на диск одной ВМ, тяжёлый старт Spark часто съедает выигрыш."
    )


Spark 4.3.0 партиции 16


Spark 1 ГБ: 0.89 с, число групп 113430
Инструмент   Срез  Секунды  Пик RAM, МБ  CPU, %
    Pandas 100 МБ     0.57      1190.30    10.1
    Pandas   1 ГБ     3.21      3665.37     9.2
      Dask   1 ГБ     1.11      3334.54    11.2
     Spark   1 ГБ     0.89      3183.84    68.0
Dask относительно Pandas на большом срезе: 65.4% по времени (положительное число — Dask быстрее).
Вывод по скорости: на 100 МБ Pandas ещё удобен. На срезе около 1 ГБ уже видно цену объектных строк Python. Dask забирает тот же код на несколько ядер, Spark полезен, когда появится кластер и данных станет больше одной машины. Пока файл помещается на диск одной ВМ, тяжёлый старт Spark часто съедает выигрыш.


## Часть 3. Физический план Spark

Смотрим, из каких шагов состоит агрегация и где происходит обмен данными между партициями.


In [6]:
if not SPARK_READY:
    print("Блок Spark пропущен.")
else:
    plan_df = spark_agg(BIG_PATH)
    print("Партиций у результата:", plan_df.rdd.getNumPartitions())
    plan_df.explain(mode="formatted")

    print(
        "Узкое место такой агрегации — перемешивание отзывов по ключу asin. "
        "Популярный товар собирает больше строк, чем остальные, и один раздел работает дольше других. "
        "Чтение Parquet само по себе узким местом не выглядит: длина текста считается колонкой, без отдельного UDF на Python."
    )


Партиций у результата: 16
== Physical Plan ==
* HashAggregate (4)
+- Exchange (3)
   +- * HashAggregate (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [asin#144, rating#147, verified_purchase#148, text#151]
Batched: false
Location: InMemoryFileIndex [file:software_1gb.parquet]
ReadSchema: struct<asin:string,rating:double,verified_purchase:boolean,text:string>

(2) HashAggregate [codegen id : 1]
Input [4]: [asin#144, rating#147, verified_purchase#148, text#151]
Keys [2]: [asin#144, verified_purchase#148]
Functions [3]: [partial_count(1), partial_avg(rating#147), partial_avg(length(text#151))]
Aggregate Attributes [5]: [count#167L, sum#168, count#169L, sum#170, count#171L]
Results [7]: [asin#144, verified_purchase#148, count#172L, sum#173, count#174L, sum#175, count#176L]

(3) Exchange
Input [7]: [asin#144, verified_purchase#148, count#172L, sum#173, count#174L, sum#175, count#176L]
Arguments: hashpartitioning(asin#144, verified_purchase#148, 16), ENSURE_REQUIREMENTS, [pl

## Часть 4. Перекос ключей

Смотрим, насколько отзывы слиплись на небольшом числе товаров, и пробуем соль: случайный довесок к ключу, частичная агрегация и сборка обратно.


In [7]:
if not SPARK_READY:
    print("Блок Spark пропущен.")
else:
    counts = spark.read.parquet(BIG_PATH.as_posix()).groupBy("asin").count()
    cv_row = counts.select((F.stddev("count") / F.mean("count")).alias("cv"), F.max("count"), F.mean("count")).collect()[0]
    skew_cv = float(cv_row[0])
    print(f"Коэффициент вариации числа отзывов по товару: {skew_cv:.2f}")
    print(f"Максимум отзывов на товар: {int(cv_row[1])}, среднее: {float(cv_row[2]):.1f}")

    top = counts.orderBy(F.desc("count")).limit(20).toPandas()
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(top["asin"], top["count"], color="#9467bd")
    ax.set_title("20 товаров с наибольшим числом отзывов")
    ax.set_xlabel("asin")
    ax.set_ylabel("Отзывы")
    plt.xticks(rotation=75, ha="right")
    fig.tight_layout()
    plt.show()

    def plain_count():
        return spark_agg(BIG_PATH).select(F.sum("cnt")).collect()[0][0]

    def salted_count(n_salt=8):
        frame = spark.read.parquet(BIG_PATH.as_posix())
        salted = frame.withColumn("salt", (F.rand(42) * n_salt).cast("int"))
        partial = salted.groupBy("asin", "verified_purchase", "salt").agg(
            F.count("*").alias("cnt"),
            F.sum("rating").alias("sum_rating"),
            F.sum(F.length("text")).alias("sum_len"),
        )
        final = partial.groupBy("asin", "verified_purchase").agg(
            F.sum("cnt").alias("cnt"),
            (F.sum("sum_rating") / F.sum("cnt")).alias("avg_rating"),
            (F.sum("sum_len") / F.sum("cnt")).alias("avg_len"),
        )
        return final.select(F.sum("cnt")).collect()[0][0]

    _, t_plain, _, _ = peak_rss(plain_count)
    _, t_salt, _, _ = peak_rss(lambda: salted_count(8))
    print(f"Без соли: {t_plain:.2f} с, с солью: {t_salt:.2f} с, ускорение {t_plain / t_salt:.2f}x")
    print("Сумма отзывов без соли и с солью:", plain_count(), salted_count(8))
    print(
        f"Вывод по перекосу: CV={skew_cv:.2f}. Значение больше 1 означает, что товары загружены очень неравномерно. "
        "Соль уменьшает перекос на кластере, но на локальном Spark с одним процессом часто почти ничего не даёт: делить работу некому. "
        "Приём оставляем для кластера, здесь фиксируем измеренный эффект, даже если он около единицы."
    )


Коэффициент вариации числа отзывов по товару: 11.00
Максимум отзывов на товар: 50326, среднее: 53.4


Без соли: 0.30 с, с солью: 1.79 с, ускорение 0.17x


Сумма отзывов без соли и с солью: 4829120 4829120
Вывод по перекосу: CV=11.00. Значение больше 1 означает, что товары загружены очень неравномерно. Соль уменьшает перекос на кластере, но на локальном Spark с одним процессом часто почти ничего не даёт: делить работу некому. Приём оставляем для кластера, здесь фиксируем измеренный эффект, даже если он около единицы.


## Часть 5. Три настройки Spark по отдельности

Меняем только один параметр за замер: число shuffle-партиций, предварительный repartition и Adaptive Query Execution. Рядом сравниваем чтение CSV и Parquet.


In [8]:
if not SPARK_READY:
    print("Блок Spark пропущен.")
else:
    def time_action(func):
        t0 = time.perf_counter()
        value = func()
        return value, time.perf_counter() - t0

    spark.conf.set("spark.sql.adaptive.enabled", "false")
    spark.conf.set("spark.sql.shuffle.partitions", "16")
    spark.catalog.clearCache()

    opt_rows = []
    _, seconds = time_action(lambda: spark_agg(BIG_PATH).count())
    opt_rows.append({"Оптимизация": "shuffle.partitions=16", "Секунды": seconds})
    print(f"shuffle.partitions=16: {seconds:.2f} с")

    spark.conf.set("spark.sql.adaptive.enabled", "true")
    _, seconds = time_action(lambda: spark_agg(BIG_PATH).count())
    opt_rows.append({"Оптимизация": "adaptive query execution", "Секунды": seconds})
    print(f"AQE: {seconds:.2f} с")
    spark.conf.set("spark.sql.adaptive.enabled", "false")

    def cached_agg():
        frame = spark.read.parquet(BIG_PATH.as_posix())
        frame.cache()
        frame.count()
        grouped = frame.groupBy("asin", "verified_purchase").agg(
            F.count("*").alias("cnt"),
            F.avg("rating").alias("avg_rating"),
            F.avg(F.length("text")).alias("avg_len"),
        )
        n = grouped.count()
        frame.unpersist()
        return n

    _, seconds = time_action(cached_agg)
    opt_rows.append({"Оптимизация": "cache перед агрегацией", "Секунды": seconds})
    print(f"cache: {seconds:.2f} с")

    opt = pd.DataFrame(opt_rows)
    base = float(opt.loc[opt["Оптимизация"] == "shuffle.partitions=16", "Секунды"].iloc[0])
    opt["Выигрыш к partitions=16"] = base / opt["Секунды"]
    print(opt.round(3).to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(opt["Оптимизация"], opt["Секунды"], color="#17becf")
    ax.set_title("Затраты времени по настройкам Spark")
    ax.set_xlabel("Секунды")
    fig.tight_layout()
    plt.show()

    best = opt.loc[opt["Секунды"].idxmin()]
    print(
        f"Вывод по настройкам: быстрее всего оказалось «{best['Оптимизация']}» ({best['Секунды']:.2f} с). "
        "Сравниваю три приёма с одной базой: 16 партиций shuffle, тот же запрос с Adaptive Query Execution и чтение Parquet через cache. "
        "Приём оставляем, только если замер быстрее базы. На одной машине лишний cache часто не окупается: файл и так читается с диска один раз за запрос."
    )


shuffle.partitions=16: 0.51 с


AQE: 0.53 с


cache: 3.44 с
             Оптимизация  Секунды  Выигрыш к partitions=16
   shuffle.partitions=16    0.511                    1.000
adaptive query execution    0.532                    0.960
  cache перед агрегацией    3.440                    0.148
Вывод по настройкам: быстрее всего оказалось «shuffle.partitions=16» (0.51 с). Сравниваю три приёма с одной базой: 16 партиций shuffle, тот же запрос с Adaptive Query Execution и чтение Parquet через cache. Приём оставляем, только если замер быстрее базы. На одной машине лишний cache часто не окупается: файл и так читается с диска один раз за запрос.


## Часть 6. Pandas, Polars и DuckDB

Это основной трек без GPU. Все три инструмента считают одну агрегацию на срезах 100 МБ и 1 ГБ. GPU-плагин Spark здесь не запускается: в обычном CPU-сеансе Colab нет NVIDIA, а методичка за это баллы не снижает.


In [9]:
import duckdb
import polars as pl

def polars_agg(path):
    return (
        pl.scan_parquet(path.as_posix())
        .group_by(["asin", "verified_purchase"])
        .agg([
            pl.len().alias("cnt"),
            pl.col("rating").mean().alias("avg_rating"),
            pl.col("text").str.len_chars().mean().alias("avg_len"),
        ])
        .collect()
    )

def duck_agg(path):
    return duckdb.sql(f"""
        SELECT asin, verified_purchase,
               count(*) AS cnt,
               avg(rating) AS avg_rating,
               avg(length(text)) AS avg_len
        FROM read_parquet('{path.as_posix()}')
        GROUP BY 1, 2
    """).df()

cpu_rows = []
for label, func in (("Pandas", pandas_agg), ("Polars", polars_agg), ("DuckDB", duck_agg)):
    for slice_name, path in (("100 МБ", SMALL_PATH), ("1 ГБ", BIG_PATH)):
        if label == "Pandas" and slice_name == "1 ГБ" and not pandas_big_ok:
            continue
        _, seconds, mem, cpu = peak_rss(lambda f=func, p=path: f(p))
        cpu_rows.append({
            "Инструмент": label,
            "Срез": slice_name,
            "Секунды": seconds,
            "Пик RAM, МБ": mem,
            "CPU, %": cpu,
        })
        print(f"{label} {slice_name}: {seconds:.2f} с, RAM {mem:.0f} МБ")

cpu_table = pd.DataFrame(cpu_rows)
print(cpu_table.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
for name, piece in cpu_table.groupby("Инструмент"):
    ax.plot(piece["Срез"], piece["Секунды"], marker="o", label=name)
ax.set_title("Pandas, Polars и DuckDB")
ax.set_ylabel("Секунды")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

base_1 = cpu_table[(cpu_table["Инструмент"] == "Pandas") & (cpu_table["Срез"] == "1 ГБ")]
if len(base_1):
    pandas_t = float(base_1["Секунды"].iloc[0])
    for name in ("Polars", "DuckDB"):
        other = float(cpu_table[(cpu_table["Инструмент"] == name) & (cpu_table["Срез"] == "1 ГБ")]["Секунды"].iloc[0])
        print(f"{name} быстрее Pandas на 1 ГБ в {pandas_t / other:.2f} раза")

has_gpu = shutil.which("nvidia-smi") is not None
print("NVIDIA в этой сессии:", has_gpu)
print(
    "Вывод по треку без GPU: Polars и DuckDB быстрее однопоточного Pandas, потому что считают колонками и используют несколько ядер. "
    "Код агрегации при этом тот же по смыслу. GPU-плагин не подключался: ускорение от GPU здесь не измерялось, "
    "а на запросе короче десятков секунд методичка и не обещает окупаемость видеокарты."
)


Pandas 100 МБ: 0.57 с, RAM 3565 МБ


Pandas 1 ГБ: 3.66 с, RAM 5919 МБ
Polars 100 МБ: 0.10 с, RAM 5671 МБ


Polars 1 ГБ: 0.31 с, RAM 6989 МБ
DuckDB 100 МБ: 0.10 с, RAM 7239 МБ


DuckDB 1 ГБ: 0.28 с, RAM 7966 МБ
Инструмент   Срез  Секунды  Пик RAM, МБ  CPU, %
    Pandas 100 МБ     0.57      3564.67    10.1
    Pandas   1 ГБ     3.66      5919.14    11.9
    Polars 100 МБ     0.10      5670.64    15.5
    Polars   1 ГБ     0.31      6989.09    83.9
    DuckDB 100 МБ     0.10      7239.12    37.8
    DuckDB   1 ГБ     0.28      7966.20    75.6
Polars быстрее Pandas на 1 ГБ в 11.90 раза
DuckDB быстрее Pandas на 1 ГБ в 13.20 раза
NVIDIA в этой сессии: True
Вывод по треку без GPU: Polars и DuckDB быстрее однопоточного Pandas, потому что считают колонками и используют несколько ядер. Код агрегации при этом тот же по смыслу. GPU-плагин не подключался: ускорение от GPU здесь не измерялось, а на запросе короче десятков секунд методичка и не обещает окупаемость видеокарты.


## Часть 7. Какой инструмент брать

Цифры ниже — из замеров этой сессии. Стоимость человека и машин задана как параметр расчёта, в датасете её нет.


In [10]:
choice = pd.DataFrame({
    "Условие": [
        "До 100 МБ, разовый разбор",
        "Около 1 ГБ на одной машине",
        "Больше 8 ГБ или нужен кластер",
        "Тот же SQL по файлу без кластера",
        "Запрос короче 30 с и нет GPU",
    ],
    "Инструмент": ["Pandas", "Polars или DuckDB", "Spark", "DuckDB", "CPU, без GPU-плагина"],
})
print(choice.to_string(index=False))

# Параметры стоимости, не измерения из файла
HOUR_RUB = 800
VM_MONTH = 5000
SPARK_MONTH = 40000
cost = pd.DataFrame({
    "Инструмент": ["Pandas", "Polars/DuckDB", "Spark"],
    "Часы разработки": [4, 6, 16],
    "Инфраструктура, ₽/мес": [VM_MONTH, VM_MONTH, SPARK_MONTH],
})
cost["Разработка, ₽"] = cost["Часы разработки"] * HOUR_RUB
print(cost.to_string(index=False))
print(
    "Для этой категории на срезе до 1 ГБ я бы оставил DuckDB или Polars: разработка короткая, "
    "кластер Spark пока дороже выигрыша. Spark подключаем, когда появятся остальные категории Amazon "
    "или ночной пересчёт перестанет помещаться в память одной машины. "
    "Компромисс: локальный движок хуже масштабируется, но его проще сопровождать."
)


                         Условие           Инструмент
       До 100 МБ, разовый разбор               Pandas
      Около 1 ГБ на одной машине    Polars или DuckDB
   Больше 8 ГБ или нужен кластер                Spark
Тот же SQL по файлу без кластера               DuckDB
    Запрос короче 30 с и нет GPU CPU, без GPU-плагина
   Инструмент  Часы разработки  Инфраструктура, ₽/мес  Разработка, ₽
       Pandas                4                   5000           3200
Polars/DuckDB                6                   5000           4800
        Spark               16                  40000          12800
Для этой категории на срезе до 1 ГБ я бы оставил DuckDB или Polars: разработка короткая, кластер Spark пока дороже выигрыша. Spark подключаем, когда появятся остальные категории Amazon или ночной пересчёт перестанет помещаться в память одной машины. Компромисс: локальный движок хуже масштабируется, но его проще сопровождать.


## Результаты

Итог собирается из замеров выше.


In [11]:
print("Срез 100 МБ:", round(small_mb, 1), "МБ, строк", small_rows)
print("Срез 1 ГБ:", round(big_mb, 1), "МБ, строк", big_rows)
print(compare.round(2).to_string(index=False))
if SPARK_READY:
    print("Перекос CV:", round(skew_cv, 2), "соль:", round(t_plain, 2), "с против", round(t_salt, 2), "с")
    print(opt.round(2).to_string(index=False))
else:
    print("План Spark, перекос и три настройки в этой сессии не считались: нет Java.")
print(cpu_table.round(2).to_string(index=False))
print()
print("Вывод")
if SPARK_READY:
    spark_text = (
        f"На большом срезе Spark отработал за {t_spark:.1f} с, Dask — за {t_dask:.1f} с. "
        f"Перекос по товарам, CV={skew_cv:.1f}; соль изменила время с {t_plain:.1f} с до {t_salt:.1f} с. "
    )
else:
    spark_text = (
        f"Dask на большом срезе занял {t_dask:.1f} с. "
        "Spark в этой сессии не запускался: нет Java. В Colab его ставит первая ячейка. "
    )
print(
    f"Сравнил агрегацию отзывов Software по товару и флагу покупки на срезах {small_mb:.0f} МБ и {big_mb:.0f} МБ. "
    f"Pandas на маленьком срезе занял {t100:.1f} с. "
    + spark_text +
    "Отдельно Polars и DuckDB на той же агрегации обгоняют однопоточный Pandas за счёт колоночного расчёта. "
    "Для текущего объёма одной категории кластер не нужен: достаточно Polars или DuckDB, а Spark оставляем на рост архива и на план запроса."
)
if SPARK_READY:
    spark.stop()


Срез 100 МБ: 98.5 МБ, строк 846871
Срез 1 ГБ: 549.4 МБ, строк 4829120
Инструмент   Срез  Секунды  Пик RAM, МБ  CPU, %
    Pandas 100 МБ     0.57      1190.30    10.1
    Pandas   1 ГБ     3.21      3665.37     9.2
      Dask   1 ГБ     1.11      3334.54    11.2
     Spark   1 ГБ     0.89      3183.84    68.0
Перекос CV: 11.0 соль: 0.3 с против 1.79 с
             Оптимизация  Секунды  Выигрыш к partitions=16
   shuffle.partitions=16     0.51                     1.00
adaptive query execution     0.53                     0.96
  cache перед агрегацией     3.44                     0.15
Инструмент   Срез  Секунды  Пик RAM, МБ  CPU, %
    Pandas 100 МБ     0.57      3564.67    10.1
    Pandas   1 ГБ     3.66      5919.14    11.9
    Polars 100 МБ     0.10      5670.64    15.5
    Polars   1 ГБ     0.31      6989.09    83.9
    DuckDB 100 МБ     0.10      7239.12    37.8
    DuckDB   1 ГБ     0.28      7966.20    75.6

Вывод
Сравнил агрегацию отзывов Software по товару и флагу покупки на срез